# Day 4 — Frozen-embedding sentiment baseline

This notebook is a thin caller for the reusable Day 4 pipeline. It uses SST-2 `train.tsv` locally, extracts frozen first-token features with DistilBERT, creates one stratified held-out split, fits Logistic Regression on the training partition, and evaluates once on the held-out partition.

## Local setup

The notebook downloads the official GLUE SST-2 archive automatically the first time its data cell runs, then reuses `data/SST-2/train.tsv`. The dataset and `baseline_results.txt` are ignored by Git. The download is about 7 MB; full feature extraction can take a long time on CPU, so start with the small smoke cells below.

In [2]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from transformers_learning import (
    DEFAULT_MODEL_NAME,
    SST2_METADATA,
    adapt_sst2_split,
    ensure_sst2_train_data,
    evaluate_logistic_regression,
    get_embeddings,
    load_model,
    load_tokenizer,
    prepare_frozen_embedding_dataset,
    save_baseline_results,
    split_frozen_embedding_dataset,
    tokenize_texts,
    train_logistic_regression,
)

## Tokenization smoke check

The tokenizer produces aligned `input_ids` and `attention_mask` tensors with shape `[batch, sequence]`. Padding lets texts of different lengths share one batch; the attention mask tells the model which positions are real tokens.

In [3]:
model_name = DEFAULT_MODEL_NAME
tokenizer = load_tokenizer(model_name)
sample_texts = [
    "A touching and beautifully acted movie.",
    "The plot was dull and predictable.",
    "I would happily watch it again.",
]
tokens = tokenize_texts(sample_texts, tokenizer, max_length=32)

print(f"input_ids shape: {tuple(tokens['input_ids'].shape)}")
print(f"attention_mask shape: {tuple(tokens['attention_mask'].shape)}")
print(tokens)

input_ids shape: (3, 9)
attention_mask shape: (3, 9)
{'input_ids': tensor([[  101,  1037,  7244,  1998, 17950,  6051,  3185,  1012,   102],
        [  101,  1996,  5436,  2001, 10634,  1998, 21425,  1012,   102],
        [  101,  1045,  2052, 11361,  3422,  2009,  2153,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])}


## Frozen first-token features

`get_embeddings` switches the base model to `eval()` and runs the forward pass under `torch.no_grad()`. Its output has shape `[n_samples, hidden_size]`: one contextual first-token vector per text. These values are features for a later classifier, not sentiment probabilities and not a claim of an optimal sentence embedding.

In [4]:
model = load_model(model_name)
sample_embeddings = get_embeddings(sample_texts, tokenizer, model, batch_size=3)

print(f"Model training mode: {model.training}")
print(f"Frozen embeddings shape: {sample_embeddings.shape}")
assert sample_embeddings.shape == (len(sample_texts), model.config.hidden_size)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model training mode: False
Frozen embeddings shape: (3, 768)


## Held-out baseline

The transformer remains frozen. Logistic Regression receives only `X_train, y_train`; `X_test, y_test` are used once for final evaluation. Macro F1 averages F1 across negative and positive classes equally, while the classification report shows support so that the score remains interpretable. Do not use the held-out result to tune this baseline.

In [5]:
sst2_train_path = ensure_sst2_train_data(
    project_root / "data" / "SST-2" / "train.tsv"
)

source_dataframe = pd.read_csv(sst2_train_path, sep="\t")
sentiment_dataframe = adapt_sst2_split(source_dataframe)
print(f"Dataset: {SST2_METADATA.identifier}")
print(f"Validated rows: {len(sentiment_dataframe)}")
print(sentiment_dataframe['label'].value_counts().sort_index())

Dataset: stanfordnlp/sst2
Validated rows: 67349
label
0    29780
1    37569
Name: count, dtype: int64


In [ ]:
embedding_dataset = prepare_frozen_embedding_dataset(
    sentiment_dataframe, tokenizer, model, batch_size=32
)
split = split_frozen_embedding_dataset(embedding_dataset)
classifier = train_logistic_regression(split)
evaluation = evaluate_logistic_regression(classifier, split)

result_path = project_root / "baseline_results.txt"
save_baseline_results(evaluation, result_path, model_name=model_name)

print(evaluation.classification_report)
print(f"Held-out macro F1: {evaluation.macro_f1:.6f}")
print(f"Saved run context to: {result_path}")